Problem Statement : Hospital Patient Data Analysis

Context:

A hospital maintains patient records including admission details, department, diagnosis, doctor, and bill amount. You have two datasets: one with patient info and another with billing details. Some patients have blank bill amounts, and there are multiple rows for the same patient due to follow-ups.
Tasks:

1.	Load the patient dataset and show summary with info().

2.	Select only the columns relevant for billing: ['PatientID', 'Department', 'Doctor', 'BillAmount'].

3.	Drop administrative columns like ['ReceptionistID', 'CheckInTime'].

4.	Use groupby to find total bill amount per department.

5.	Remove duplicate patient records based on PatientID.

6.	Fill missing BillAmount values with the mean bill amount.

7.	Merge the billing dataset with patient dataset on PatientID.

8.	Concatenate an additional DataFrame that contains new patients for the current week (row-wise).

9.	Concatenate new billing category columns like ['InsuranceCovered', 'FinalAmount'] (column-wise).

Expected Outcome:

•	Final cleaned dataset with accurate billing info.

•	All missing values handled, merged dataset across PatientID.

•	Ability to perform further analytics on department-wise revenue or doctor performance.


In [ ]:
#importing libraries and loading dataset
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
patient_df = pd.read_csv("Patient_Data.csv")
billing_df = pd.read_csv("Billing_Data.csv")


In [ ]:
#summary of patient dataset
patient_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PatientID       6 non-null      int64  
 1   Name            6 non-null      object 
 2   Department      6 non-null      object 
 3   Doctor          6 non-null      object 
 4   BillAmount      4 non-null      float64
 5   ReceptionistID  6 non-null      int64  
 6   CheckInTime     6 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 468.0+ bytes


In [ ]:
#selecting columns relevant for billing
billing_relevant_df = patient_df[['PatientID', 'Department', 'Doctor', 'BillAmount']]
billing_relevant_df

,PatientID,Department,Doctor,BillAmount
0,101,Cardiology,Dr. Smith,5000.0
1,102,Neurology,Dr. John,NaN
2,103,Orthopedics,Dr. Lee,7500.0
3,104,Cardiology,Dr. Smith,6200.0
4,105,Dermatology,Dr. Rose,NaN
5,101,Cardiology,Dr. Smith,5000.0


In [ ]:
#total bill amount per department using groupby function
department_bill_total = billing_relevant_df.groupby('Department')['BillAmount'].sum()
department_bill_total

,BillAmount
Department,
Cardiology,16200.0
Dermatology,0.0
Neurology,0.0
Orthopedics,7500.0


In [ ]:
#removing dupplicates based on patient id
billing_relevant_df = billing_relevant_df.drop_duplicates(subset=['PatientID'])
billing_relevant_df

,PatientID,Department,Doctor,BillAmount
0,101,Cardiology,Dr. Smith,5000.0
1,102,Neurology,Dr. John,NaN
2,103,Orthopedics,Dr. Lee,7500.0
3,104,Cardiology,Dr. Smith,6200.0
4,105,Dermatology,Dr. Rose,NaN


In [ ]:
#filling missing values with mean values
mean = billing_relevant_df['BillAmount'].mean()
billing_relevant_df["BillAmount"] = billing_relevant_df["BillAmount"].fillna(mean)
billing_relevant_df

,PatientID,Department,Doctor,BillAmount
0,101,Cardiology,Dr. Smith,5000.000000
1,102,Neurology,Dr. John,6233.333333
2,103,Orthopedics,Dr. Lee,7500.000000
3,104,Cardiology,Dr. Smith,6200.000000
4,105,Dermatology,Dr. Rose,6233.333333


In [ ]:
#merging the billing n patient dataset using patient id
merged_df = pd.merge(patient_df, billing_df, on='PatientID', how = 'inner')
merged_df

,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0,1,2023-01-10 09:00,2000,3000
1,102,Bob,Neurology,Dr. John,NaN,2,2023-01-11 10:30,1500,3500
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,1,2023-01-12 11:00,2500,5000
3,104,David,Cardiology,Dr. Smith,6200.0,3,2023-01-13 12:00,3000,3200
4,105,Eva,Dermatology,Dr. Rose,NaN,2,2023-01-14 08:45,1000,4000
5,101,Alice,Cardiology,Dr. Smith,5000.0,1,2023-01-10 09:00,2000,3000


In [ ]:
#creating a new df for new patient and cancatenate
new_patients_df = pd.DataFrame({
    'PatientID': [106, 107],
    'Department': ['Cardiology', 'Neurology'],
    'Doctor': ['Dr. Smith', 'Dr. John'],
    'BillAmount': [7000, 8000]
})

updated_patient_df = pd.concat([billing_relevant_df, new_patients_df],axis=0,ignore_index=True)
updated_patient_df


,PatientID,Department,Doctor,BillAmount
0,101,Cardiology,Dr. Smith,5000.000000
1,102,Neurology,Dr. John,6233.333333
2,103,Orthopedics,Dr. Lee,7500.000000
3,104,Cardiology,Dr. Smith,6200.000000
4,105,Dermatology,Dr. Rose,6233.333333
5,106,Cardiology,Dr. Smith,7000.000000
6,107,Neurology,Dr. John,8000.000000


In [ ]:
#creating new billing columns
new_billing_columns = pd.DataFrame({'InsuranceCovered': [True] * len(updated_patient_df),'FinalAmount': updated_patient_df['BillAmount'] * 0.8})
new_billing_columns


,InsuranceCovered,FinalAmount
0,True,4000.000000
1,True,4986.666667
2,True,6000.000000
3,True,4960.000000
4,True,4986.666667
5,True,5600.000000
6,True,6400.000000


In [ ]:
#concatenating new billing columns to the dataset
final_patient_df = pd.concat([updated_patient_df.reset_index(drop=True), new_billing_columns],axis=1)

In [ ]:
final_patient_df

,PatientID,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount
0,101,Cardiology,Dr. Smith,5000.000000,True,4000.000000
1,102,Neurology,Dr. John,6233.333333,True,4986.666667
2,103,Orthopedics,Dr. Lee,7500.000000,True,6000.000000
3,104,Cardiology,Dr. Smith,6200.000000,True,4960.000000
4,105,Dermatology,Dr. Rose,6233.333333,True,4986.666667
5,106,Cardiology,Dr. Smith,7000.000000,True,5600.000000
6,107,Neurology,Dr. John,8000.000000,True,6400.000000
